# 03. Анализ пространства эмбеддингов MARLIN

**Цель:** исследовать структуру признакового пространства MARLIN:
- Визуализация t-SNE, цвета: по классу и по субъекту
- PCA: объяснённая дисперсия
- Важность признаков Random Forest (топ-30 измерений эмбеддинга)
- Сравнение MARLIN vit_small vs vit_base по метрикам (итоговый рисунок)

---

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from sklearn.manifold        import TSNE
from sklearn.decomposition   import PCA
from sklearn.preprocessing   import StandardScaler
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline        import Pipeline

RANDOM_STATE = 42
N_SPLITS     = 5
EMB_BASE_SMALL = Path("/content/drive/MyDrive/deception_marlin_group_split/embeddings_vit_small")
EMB_BASE_BASE  = Path("/content/drive/MyDrive/deception_marlin_group_split/embeddings_vit_base")

plt.rcParams.update({
    "figure.dpi":      120,
    "font.family":     "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})
COLORS = {"truthful": "#4C72B0", "deceptive": "#DD8452"}
print("Готово.")

## 1. Загрузка эмбеддингов vit_base

In [ ]:
rows = []
for label_name, label in [("truthful", 1), ("deceptive", 0)]:
    for npy in sorted((EMB_BASE_BASE / label_name).glob("*.npy")):
        subject_id = npy.stem.split("_")[0]
        rows.append({
            "video":          npy.stem,
            "label":          label,
            "label_name":     label_name,
            "subject_id":     subject_id,
            "embedding_path": npy,
        })

full_df = pd.DataFrame(rows)
X = np.stack([np.load(p) for p in full_df["embedding_path"]])
y = full_df["label"].to_numpy()
groups = full_df["subject_id"].to_numpy()

print(f"X.shape: {X.shape}  (videos × embedding_dim)")
print(f"Классов: truthful={y.sum()}, deceptive={(y==0).sum()}")

## 2. PCA: объяснённая дисперсия

Позволяет оценить, насколько компактно пространство эмбеддингов и достаточно ли 50/100 компонент для описания данных.

In [ ]:
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

pca_full = PCA(random_state=RANDOM_STATE).fit(X_sc)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(np.arange(1, len(cumvar) + 1), cumvar, lw=1.8, color="#4C72B0")
ax.fill_between(np.arange(1, len(cumvar) + 1), cumvar, alpha=0.12, color="#4C72B0")

for n_comp, color in [(50, "#e07070"), (100, "#e0a040"), (200, "#40a040")]:
    if n_comp <= len(cumvar):
        ax.axvline(n_comp, color=color, linestyle="--", lw=1.2)
        ax.text(n_comp + 3, cumvar[n_comp - 1] - 0.04,
                f"n={n_comp}\n{cumvar[n_comp-1]:.0%}",
                color=color, fontsize=8)

ax.set_xlabel("Число главных компонент", fontsize=11)
ax.set_ylabel("Накопленная объяснённая дисперсия", fontsize=11)
ax.set_title("PCA: накопленная объяснённая дисперсия\nэмбеддингов MARLIN vit_base (768-мерные)", fontsize=11, pad=8)
ax.set_ylim(0, 1.05)
ax.set_xlim(0, X.shape[1])

plt.tight_layout()
plt.savefig("fig_pca_variance.png", bbox_inches="tight")
plt.show()
print("Сохранено: fig_pca_variance.png")

## 3. t-SNE: визуализация пространства эмбеддингов

Двойная визуализация: слева цвет по **классу**, справа — по **субъекту**.  
Если правый график показывает чёткие кластеры по субъектам, это объясняет, почему наивное разделение даёт завышенные результаты.

In [ ]:
X_pca50 = PCA(n_components=50, random_state=RANDOM_STATE).fit_transform(X_sc)
tsne = TSNE(n_components=2, perplexity=15, random_state=RANDOM_STATE,
            n_iter=2000, learning_rate="auto", init="pca")
X_2d = tsne.fit_transform(X_pca50)
print("t-SNE выполнен.")

In [ ]:
unique_subjects = sorted(full_df["subject_id"].unique())
subj_cmap = plt.cm.get_cmap("tab20", len(unique_subjects))
subj_color_map = {s: subj_cmap(i) for i, s in enumerate(unique_subjects)}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Слева: по классу ---
ax = axes[0]
for label_name, label, marker in [("truthful", 1, "o"), ("deceptive", 0, "s")]:
    mask = y == label
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=COLORS[label_name], marker=marker,
        s=45, alpha=0.75, edgecolors="white", linewidths=0.4,
        label=label_name.capitalize()
    )
ax.set_title("t-SNE: окраска по классу", fontsize=11, pad=8)
ax.legend(fontsize=9, markerscale=1.2)
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# --- Справа: по субъекту ---
ax = axes[1]
for i, subj in enumerate(unique_subjects):
    mask = groups == subj
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=[subj_color_map[subj]], s=45, alpha=0.8,
        edgecolors="white", linewidths=0.3
    )
    # Подписываем субъектов с >6 клипами
    if mask.sum() > 6:
        cx, cy = X_2d[mask, 0].mean(), X_2d[mask, 1].mean()
        ax.annotate(subj, (cx, cy), fontsize=6, ha="center",
                    color="#333", alpha=0.8)

ax.set_title("t-SNE: окраска по субъекту", fontsize=11, pad=8)
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("")
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle("Эмбеддинги MARLIN vit_base: пространство признаков (t-SNE, perplexity=15)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("fig_tsne_dual.png", bbox_inches="tight")
plt.show()
print("Сохранено: fig_tsne_dual.png")

## 4. Важность признаков Random Forest

Какие измерения 768-мерного эмбеддинга наиболее дискриминативны для задачи детекции лжи?

In [ ]:
# Лучшая стратегия — std_only: используем отклонение от среднего по признакам
X_std = np.abs(X - X.mean(axis=0))

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                             random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_std, y)
importances = rf.feature_importances_

top_n = 30
top_idx = np.argsort(importances)[::-1][:top_n]
top_imp = importances[top_idx]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(top_n), top_imp,
       color=plt.cm.Blues(np.linspace(0.4, 0.9, top_n)[::-1]),
       edgecolor="white", linewidth=0.5)
ax.set_xticks(np.arange(top_n))
ax.set_xticklabels([f"dim {i}" for i in top_idx], rotation=75, ha="right", fontsize=7)
ax.set_xlabel("Измерение эмбеддинга (индекс)", fontsize=10)
ax.set_ylabel("Важность признака (Gini)", fontsize=10)
ax.set_title(
    f"Топ-{top_n} наиболее важных измерений MARLIN vit_base (стратегия std_only)\n"
    f"Random Forest, обучение на всех данных",
    fontsize=10, pad=8
)

plt.tight_layout()
plt.savefig("fig_feature_importance.png", bbox_inches="tight")
plt.show()
print("Сохранено: fig_feature_importance.png")
print(f"Топ-10 измерений: {top_idx[:10].tolist()}")
print(f"Суммарная важность топ-{top_n}: {top_imp.sum():.3f}")

## 5. Сравнение MARLIN vit_small vs vit_base

Итоговый рисунок для диплома: как размерность латентного пространства и ёмкость энкодера влияют на качество классификации.

In [ ]:
# Результаты из экспериментов (вставьте свои значения из notebook 02)
configs = [
    "MARLIN vit_small\n(dim=384, mean+std)",
    "MARLIN vit_base\n(dim=768, mean+std)",
    "MARLIN vit_base\n(dim=768, std_only)",
]
bal_acc = [0.487, 0.587, 0.668]
auc_    = [0.544, 0.618, 0.628]
std_ba  = [0.080, 0.095, 0.110]

x = np.arange(len(configs))
w = 0.32
palette = ["#70a0d0", "#d07070"]

fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(x - w/2, bal_acc, w, label="Balanced Accuracy",
            color=palette[0], yerr=std_ba, capsize=5,
            error_kw=dict(elinewidth=1.3, ecolor="#444"))
b2 = ax.bar(x + w/2, auc_,    w, label="AUC-ROC",
            color=palette[1], alpha=0.85)

for bar, v in zip(list(b1) + list(b2), bal_acc + auc_):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)

ax.axhline(0.5, color="grey", linestyle="--", lw=1.2, label="Случайная модель")
ax.set_xticks(x)
ax.set_xticklabels(configs, fontsize=9)
ax.set_ylabel("Значение метрики", fontsize=10)
ax.set_title(
    "Сравнение конфигураций MARLIN по качеству классификации\n"
    "(5-fold StratifiedGroupKFold, датасет Real-Life Trial)",
    fontsize=10, pad=8
)
ax.set_ylim(0.3, 0.82)
ax.legend(fontsize=9)

# Стрелка прогресса
for i in range(len(configs) - 1):
    ax.annotate("", xy=(i + 1 - w/2, bal_acc[i+1] + 0.03),
                xytext=(i - w/2, bal_acc[i] + 0.03),
                arrowprops=dict(arrowstyle="->", color=palette[0], lw=1.2))

plt.tight_layout()
plt.savefig("fig_marlin_comparison.png", bbox_inches="tight")
plt.show()
print("Сохранено: fig_marlin_comparison.png")

## 6. Распределение значений std-признаков по классам

Показывает, есть ли различия в распределении вариативности эмбеддинга между правдивыми и ложными видео.

In [ ]:
X_std = np.abs(X - X.mean(axis=0))

# Суммарная вариативность (L1-норма std-вектора) как скалярная характеристика
variability = X_std.sum(axis=1)

fig, ax = plt.subplots(figsize=(6, 4))
for label_name, label in [("truthful", 1), ("deceptive", 0)]:
    mask = y == label
    ax.hist(variability[mask], bins=20, alpha=0.65,
            color=COLORS[label_name], label=label_name.capitalize(),
            edgecolor="white", linewidth=0.5, density=True)

for label_name, label in [("truthful", 1), ("deceptive", 0)]:
    mask = y == label
    ax.axvline(variability[mask].mean(), color=COLORS[label_name],
               linestyle="--", lw=1.5, alpha=0.8)

ax.set_xlabel("Суммарная вариативность эмбеддинга (L1-норма std-вектора)", fontsize=10)
ax.set_ylabel("Плотность", fontsize=10)
ax.set_title(
    "Распределение вариативности MARLIN vit_base по классам\n"
    "(пунктир — среднее значение класса)",
    fontsize=10, pad=8
)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("fig_variability_distribution.png", bbox_inches="tight")
plt.show()
print("Сохранено: fig_variability_distribution.png")

for label_name, label in [("truthful", 1), ("deceptive", 0)]:
    mask = y == label
    print(f"{label_name}: mean={variability[mask].mean():.2f}, std={variability[mask].std():.2f}")

---
**Выводы:**
- t-SNE показывает, что пространство эмбеддингов структурировано **по субъектам**, а не по классам — это объясняет высокую дисперсию результатов при разделении по субъектам
- Важность признаков RF распределена по многим измерениям эмбеддинга — нет нескольких доминирующих, что типично для трансформерных представлений
- Переход от vit_small к vit_base даёт прирост Balanced Accuracy на ~10 п.п., а оптимальная стратегия агрегирования добавляет ещё ~8 п.п.